In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("../../")

import pyaldata as pyal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

from tools.reports.report_initial import run_initial_report
from tools.params import Params, colors
from tools.dsp.preprocessing import preprocess
import tools.viz.mean_firing as firing
import tools.viz.dimensionality as dim
import tools.viz.utilityTools as vizutils
import tools.decoding.rrr as rrr
import tools.decoding.decodeTools as decutils
import tools.dataTools as dt




In [ ]:
# Files 
session = 'M062_2025_03_21_14_00'
data_dir = f"/data/bnd-data/raw/M062/{session}"

areas=["MOp", "SSp", "CP", "VAL"]
df = pyal.load_pyaldata(data_dir)

In [ ]:
df_ = preprocess(df, only_trials=False, repair_time_varying_fields=['MotSen1_X', 'MotSen1_Y'])

In [ ]:
pyal.get_time_varying_fields(df_)

In [ ]:
## trial design
area="VAL"
trial = 23
tr = pyal.concat_trials(df_, f"{area}_spikes", trial_indices=[trial-1, trial, trial+1])

mot_y = pyal.concat_trials(df_, "values_MotSen1_Y", trial_indices=[trial-1, trial, trial+1])
mot_x = pyal.concat_trials(df_, "values_MotSen1_X", trial_indices=[trial-1, trial, trial+1])

left_elbow_angle = pyal.concat_trials(df_, "left_elbow_angle", trial_indices=[trial-1, trial, trial+1])
right_elbow_angle = pyal.concat_trials(df_, "right_elbow_angle", trial_indices=[trial-1, trial, trial+1])

with plt.style.context('seaborn-v0_8-bright'):
    sns.set_theme(context='talk', style='ticks')
    fig, ax = plt.subplots(3, 1, figsize=(7, 10), sharex='all', gridspec_kw={'height_ratios': [1, 4, 1], 'hspace': 0.1})
    times = np.arange(tr.shape[0]) * 0.03
    im = ax[1].imshow(tr.T[:], cmap=vizutils.create_cmap_from_area('all'), origin="lower", aspect="auto")
    xticks = [0, df_.CP_rates.values[trial-1].shape[0]+1, df_.CP_rates.values[trial-1].shape[0]+1+66, df_.CP_rates.values[trial-1].shape[0]+1 + 200]  # 6 ticks
    ax[1].set_xticks(xticks)
    ax[1].set_xticklabels([f"{(x * Params.BIN_SIZE)-6:.1f}" for x in xticks])

    behav = np.zeros(tr.shape[0])
    ax[0].plot(behav, 'k')
    ax[0].vlines(x=df_.CP_rates.values[trial-1].shape[0]+1, ymin=-1, ymax=1, color='k')
    ax[0].vlines(x=df_.CP_rates.values[trial-1].shape[0]+1 + 66, ymin=-1, ymax=1, color='k')
    ax[0].vlines(x=df_.CP_rates.values[trial-1].shape[0]+1 + 70, ymin=-1, ymax=1, color='k')
    ax[0].vlines(x=df_.CP_rates.values[trial-1].shape[0]+1 + 200, ymin=-1, ymax=1, color='k')
    ax[0].set_yticks([])
    ax[0].set_ylim([-2, 2])
    for spine in ax[0].spines.values():
        spine.set_visible(False)



    ax[1].set_yticks([])
    ax[1].set_ylabel('Neurons')


    # ax[2].plot(mot_y)
    # ax[2].plot(mot_x)
    ax[2].set_yticks([])
    ax[2].set_ylabel('Mot. Sensors')
    ax[2].set_xlabel('Time (s)')


    ax[2].plot(left_elbow_angle)
    ax[2].plot(right_elbow_angle)


plt.show()


In [ ]:
pyal.get_time_varying_fields(df, ref_field='left_elbow_angle')

In [ ]:
df.right_wrist_angle.values[-1].shape

In [ ]:
np.concatenate(df.right_wrist_angle.values[:]).shape

In [ ]:
df.CP_spikes.values[-1].shape

In [ ]:
df_trials = pyal.select_trials(df_, df_.trial_name == 'trial')
trials = 6
probe_1 = np.concatenate([pyal.concat_trials(df_trials, 'MOp_spikes', np.arange(trials)), pyal.concat_trials(df_trials, 'CP_spikes', np.arange(trials))], axis=1)
probe_2 = np.concatenate([pyal.concat_trials(df_trials, 'SSp_spikes', np.arange(trials)), pyal.concat_trials(df_trials, 'VAL_spikes', np.arange(trials))], axis=1)



In [ ]:
probe_1.shape

In [ ]:
spike_times_per_neuron_1 = [np.where(row)[0] for row in probe_1.T]
spike_times_per_neuron_2 = [np.where(row)[0] for row in probe_2.T]

with plt.style.context('seaborn-v0_8-bright'):
    sns.set_theme(context='talk', style='ticks')
    fig, ax = plt.subplots(2, 1, figsize=(10, 10), sharex='all')
    ax[0].eventplot(spike_times_per_neuron_1, orientation='horizontal', linelengths=0.8, colors='black')
    ax[1].eventplot(spike_times_per_neuron_2, orientation='horizontal', linelengths=0.8, colors='black')
    ax[0].set_yticks([])
    ax[0].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticks([])

    for spine in ax[0].spines.values():
        spine.set_visible(False)
    for spine in ax[1].spines.values():
        spine.set_visible(False)
    ax[0].invert_yaxis()  # Optional: neuron 0 at top
    ax[1].invert_yaxis()  # Optional: neuron 0 at top

    # ax.set_xlim(0, 200)
plt.show()

In [ ]:
spike_times_per_neuron = [np.where(row)[0] for row in probe_2.T]
with plt.style.context('seaborn-v0_8-bright'):
    sns.set_theme(context='talk', style='ticks')
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.eventplot(spike_times_per_neuron, orientation='horizontal', linelengths=0.8, colors='black')

    ax.set_yticks([])
    ax.set_xticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.invert_yaxis()  # Optional: neuron 0 at top
    # ax.set_xlim(0, 200)
plt.show()

In [ ]:
probe_1.shape

In [ ]:
probe_2.shape